# 01. Simplest BattINFO Flow: One Cell Type From JSON

This notebook shows the smallest useful BattINFO workflow.

You will:

- generate a hand-editable cell-type draft JSON
- load it into a `Workspace`
- canonize it into a BattINFO `cell-type` record
- query the saved result locally

Everything writes under `.battinfo/notebooks/01-cell-type-draft`.


In [ ]:
from pathlib import Path
import json

from battinfo import Workspace, quantity, template_cell_type_draft


## 1. Create a workspace and a draft JSON file

The draft JSON is intentionally simple. It does not include canonical fields like `id`, `short_id`, or `identifier`.


In [ ]:
workspace = Workspace(root=Path('.battinfo/notebooks/01-cell-type-draft'), clean=True)
draft_path = workspace.root / 'drafts' / 'GOOGLE__G20M7.json'
draft_path.parent.mkdir(parents=True, exist_ok=True)

draft = template_cell_type_draft(
    manufacturer='Google',
    model_name='G20M7',
    chemistry='Li-ion',
    format='prismatic',
    size_code='P6/65/75',
    iec_code='ICP6/65/75',
    country_of_origin='Vietnam',
    year=2025,
)
draft['positive_electrode_basis'] = 'LCO'
draft['negative_electrode_basis'] = 'Graphite'
draft['specs'] = {
    'nominal_capacity': quantity(4.97, 'Ah'),
    'rated_capacity': quantity(4.835, 'Ah'),
    'nominal_voltage': quantity(3.9, 'V'),
    'charging_voltage': quantity(4.5, 'V'),
    'discharging_cutoff_voltage': quantity(3.0, 'V'),
    'mass': quantity(63, 'g'),
    'width': quantity(65, 'mm'),
    'length': quantity(75, 'mm'),
    'thickness': quantity(6, 'mm'),
}
draft['provenance'] = {'source_file': 'g20m7-label-notes.md'}
draft['comment'] = 'Manual draft created by notebook 01.'

draft_path.write_text(json.dumps(draft, indent=2), encoding='utf-8')

{'draft_path': draft_path, 'draft_keys': list(draft.keys())}


## 2. Load the draft into the workspace

At this point the object is still a draft. BattINFO has not minted a canonical ID yet.


In [ ]:
cell_type = workspace.load_cell_type(draft_path)

{
    'manufacturer': cell_type.manufacturer,
    'model': cell_type.model,
    'id_before_save': cell_type.id,
    'country_of_origin': cell_type.country_of_origin,
    'year': cell_type.year,
}


## 3. Save and inspect the canonical BattINFO record

`workspace.save()` canonizes the draft. This is where BattINFO mints the persistent IRI and writes the canonical `cell-type` JSON record.


In [ ]:
save_result = workspace.save(validation_policy='strict')
canonical_path = workspace.source_root / 'cell-type' / f"cell-type-{cell_type.id.rsplit('/', 1)[-1]}.json"
canonical_record = json.loads(canonical_path.read_text(encoding='utf-8'))

{
    'cell_type_id': cell_type.id,
    'canonical_path': canonical_path,
    'saved_cell_type_count': save_result['index']['cell_type_count'],
    'country_of_origin': canonical_record['product']['countryOfOrigin'],
    'year': canonical_record['product']['year'],
}


In [ ]:
workspace.query_cell_types(manufacturer='Google', format='prismatic')
